# [Lab2] 모델 훈련 with SageMaker Training

이 노트북에서는 SageMaker Training Job을 사용하여 XGBoost 모델을 훈련합니다.

## 주요 내용
- SageMaker Built-in 알고리즘을 사용한 모델 훈련
- 다양한 하이퍼파라미터 설정으로 모델 비교
- MLflow를 통한 실험 추적 및 모델 등록
- Script Mode를 사용한 커스텀 훈련
- 자동 로깅 기능 활용

## 1. 환경 설정 및 변수 로드

In [1]:
# 이전 노트북에서 저장한 변수들 로드
%store -r

# [2026-06-20 수정] 0-setup 을 먼저 실행하지 않으면 %store 복원이 비어 NameError 가 납니다.
#   필수 변수가 없으면 어디서 막혔는지 친절히 안내합니다.
_required = ['bucket', 'prefix', 'train_path', 'validation_path', 'experiment_name']
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"필수 변수 누락: {_missing}\n"
        "👉 같은 JupyterLab 커널에서 0-setup.ipynb 와 1-preprocessing.ipynb 를 먼저 실행해주세요."
    )

print("✅ 저장된 변수들을 로드했습니다.")
print(f"   - S3 버킷: {bucket}")
print(f"   - 훈련 데이터 경로: {train_path}")
print(f"   - 검증 데이터 경로: {validation_path}")
print(f"   - 실험 이름: {experiment_name}")

✅ 저장된 변수들을 로드했습니다.
   - S3 버킷: csv-file-store-db634350
   - 훈련 데이터 경로: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/train
   - 검증 데이터 경로: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/validation
   - 실험 이름: end-to-end-experiment-20-18-00-43


In [2]:
# 필수 라이브러리 임포트
import sagemaker
import boto3
import mlflow
import os
from time import gmtime, strftime
from sagemaker.xgboost.estimator import XGBoost
from IPython.display import Javascript

# AWS 세션 초기화
boto_session = boto3.Session()
sess = sagemaker.Session()

print("✅ 라이브러리 임포트 및 세션 초기화 완료")

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
✅ 라이브러리 임포트 및 세션 초기화 완료


## 2. 훈련 데이터 준비

전처리된 데이터를 SageMaker Training Job에서 사용할 수 있도록 설정합니다.

In [3]:
# XGBoost 컨테이너 이미지 URI 가져오기
container = sagemaker.image_uris.retrieve(
    region=boto3.Session().region_name, 
    framework='xgboost', 
    version='1.7-1'
)

print(f"✅ XGBoost 컨테이너 이미지: {container}")

✅ XGBoost 컨테이너 이미지: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1


In [4]:
# 훈련 입력 데이터 설정
s3_input_train = sagemaker.inputs.TrainingInput(
    s3_data=train_path, 
    content_type='csv'
)
s3_input_validation = sagemaker.inputs.TrainingInput(
    s3_data=validation_path, 
    content_type='csv'
)

training_inputs = {
    'train': s3_input_train, 
    'validation': s3_input_validation
}

print("✅ 훈련 입력 데이터 설정 완료")
print(f"   - 훈련 데이터: {train_path}")
print(f"   - 검증 데이터: {validation_path}")

✅ 훈련 입력 데이터 설정 완료
   - 훈련 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/train
   - 검증 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/validation


## 3. MLflow 실험 설정

모델 훈련 과정을 추적하기 위한 MLflow 설정을 수행합니다.

In [5]:
# 프로젝트 및 MLflow 설정
from sagemaker_studio import Project

project = Project()
# 0-setup 노트북에서 %store 한 mlflow_arn 사용
# (project.mlflow_tracking_server_arn 은 environment 이름이 정확히 'MLExperiments'가
#  아니면 실패하므로 사용하지 않음)
arn = mlflow_arn
role = project.iam_role
domain_id = project.domain_id
project_id= project.id

mlflow.set_tracking_uri(arn)
mlflow.set_experiment(experiment_name=experiment_name)

# 사용자 프로필 정보 설정
user_profile_name = os.getenv('USER', 'sagemaker-user')

print(f"✅ MLflow 실험 설정 완료")
print(f"   - 추적 URI: {arn}")
print(f"   - 실험 이름: {experiment_name}")
print(f"   - 사용자: {user_profile_name}")

✅ MLflow 실험 설정 완료
   - 추적 URI: arn:aws:sagemaker:us-west-2:975050309668:mlflow-tracking-server/tracking-server-4gwjxwgbrut11s-6rbrrb5bcww7ww-dev
   - 실험 이름: end-to-end-experiment-20-18-00-43
   - 사용자: sagemaker-user


## 4. Built-in 알고리즘을 사용한 모델 훈련

SageMaker의 Built-in XGBoost 알고리즘을 사용하여 두 가지 다른 하이퍼파라미터 설정으로 모델을 훈련합니다.

In [6]:
# [2026-06-20 수정] 하이퍼파라미터를 dict(model1_hyperparams)로 단일 정의.
#   기존엔 set_hyperparameters 값과 아래 MLflow log_params 값을 따로 손입력해
#   한쪽만 고치면 잘못된 값이 기록되던 문제를 방지.
# 첫 번째 모델 설정 (보수적인 하이퍼파라미터)
xgb1 = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role, 
    instance_count=1, 
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output/',
    base_job_name='conservative-xgb-training',
    sagemaker_session=sess
)

# 하이퍼파라미터를 dict 로 한 번만 정의 → 훈련 설정과 MLflow 로깅이 항상 일치
model1_hyperparams = {
    "max_depth": 3,
    "eta": 0.5,
    "gamma": 4,
    "eval_metric": "auc",
    "min_child_weight": 6,
    "subsample": 0.8,
    "verbosity": 0,
    "objective": "binary:logistic",
    "num_round": 50,
}
xgb1.set_hyperparameters(**model1_hyperparams)

print("✅ 첫 번째 모델 (보수적 설정) 구성 완료")
print(f"   - 학습률(eta): {model1_hyperparams['eta']}")
print(f"   - 최대 깊이: {model1_hyperparams['max_depth']}")
print(f"   - 라운드 수: {model1_hyperparams['num_round']}")

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
✅ 첫 번째 모델 (보수적 설정) 구성 완료
   - 학습률(eta): 0.5
   - 최대 깊이: 3
   - 라운드 수: 50


In [7]:
# [2026-06-20 수정] 하이퍼파라미터를 dict(model2_hyperparams)로 단일 정의 (model1 과 동일 취지).
# 두 번째 모델 설정 (적극적인 하이퍼파라미터)
xgb2 = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role, 
    instance_count=1, 
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output/',
    base_job_name='aggressive-xgb-training',
    sagemaker_session=sess
)

# 하이퍼파라미터를 dict 로 한 번만 정의 → 훈련 설정과 MLflow 로깅이 항상 일치
model2_hyperparams = {
    "max_depth": 3,
    "eta": 0.1,
    "gamma": 2,
    "eval_metric": "auc",
    "min_child_weight": 3,
    "subsample": 0.4,
    "verbosity": 0,
    "objective": "binary:logistic",
    "num_round": 100,
}
xgb2.set_hyperparameters(**model2_hyperparams)

print("✅ 두 번째 모델 (적극적 설정) 구성 완료")
print(f"   - 학습률(eta): {model2_hyperparams['eta']}")
print(f"   - 최대 깊이: {model2_hyperparams['max_depth']}")
print(f"   - 라운드 수: {model2_hyperparams['num_round']}")

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
✅ 두 번째 모델 (적극적 설정) 구성 완료
   - 학습률(eta): 0.1
   - 최대 깊이: 3
   - 라운드 수: 100


## 5. MLflow 실험 시작 및 모델 훈련

MLflow 실험을 시작하고 두 모델을 훈련합니다.

In [8]:
# 실행 이름 생성
run_suffix = strftime('%d-%H-%M-%S', gmtime())
run_name = f"model-training-{run_suffix}"

# 이미 활성화된 run 이 있으면 정리 (셀 재실행/이전 run 잔존 시 'already active' 에러 방지)
if mlflow.active_run() is not None:
    mlflow.end_run()

# MLflow 실행 시작
run_id = mlflow.start_run(
    run_name=run_name, 
    description="SageMaker Built-in XGBoost를 사용한 모델 훈련"
).info.run_id

print(f"✅ MLflow 실행 시작: {run_name}")
print(f"   - 실행 ID: {run_id}")

✅ MLflow 실행 시작: model-training-20-18-35-27
   - 실행 ID: b2334fdb9a5249f1a6b5f329847c5500


In [9]:
# 첫 번째 모델 훈련 시작 (비동기)
print("🚀 첫 번째 모델 훈련 시작 (백그라운드)...")
xgb1.fit(training_inputs, wait=False, logs=False)

print(f"✅ 첫 번째 모델 훈련 작업 시작: {xgb1.latest_training_job.name}")

🚀 첫 번째 모델 훈련 시작 (백그라운드)...
✅ 첫 번째 모델 훈련 작업 시작: conservative-xgb-training-2026-06-20-18-35-27-680


In [10]:
# 두 번째 모델 훈련 시작 (동기)
print("🚀 두 번째 모델 훈련 시작 (대기)...")
xgb2.fit(training_inputs, wait=True, logs=False)
print(f"✅ 두 번째 모델 훈련 완료: {xgb2.latest_training_job.name}")

# [2026-06-20 수정] model1 은 위에서 wait=False(백그라운드)로 시작했으므로 여기서 완료를 명시적으로 대기.
#   이 대기가 없으면 아래 로깅 단계에서 아직 생성되지 않은 model1 아티팩트를 내려받다 간헐적으로 실패합니다.
print("⏳ 첫 번째 모델(백그라운드) 완료 대기 중...")
xgb1.latest_training_job.wait()
print(f"✅ 첫 번째 모델 훈련 완료: {xgb1.latest_training_job.name}")

🚀 두 번째 모델 훈련 시작 (대기)...

2026-06-20 18:35:33 Starting - Starting the training job..
2026-06-20 18:35:47 Starting - Preparing the instances for training....
2026-06-20 18:36:11 Downloading - Downloading input data.........
2026-06-20 18:37:02 Downloading - Downloading the training image............
2026-06-20 18:38:07 Training - Training image download completed. Training in progress...
2026-06-20 18:38:23 Uploading - Uploading generated training model.
2026-06-20 18:38:36 Completed - Training job completed
✅ 두 번째 모델 훈련 완료: aggressive-xgb-training-2026-06-20-18-35-29-287
⏳ 첫 번째 모델(백그라운드) 완료 대기 중...
2026-06-20 18:38:34 Starting - Preparing the instances for training
2026-06-20 18:38:34 Downloading - Downloading the training image
2026-06-20 18:38:34 Training - Training image download completed. Training in progress.
2026-06-20 18:38:34 Uploading - Uploading generated training model
2026-06-20 18:38:34 Completed - Training job completed/miniconda3/lib/python3.9/site-packages/sagemaker_con

## ⏳ 두 모델 훈련 대기

`model1` 은 `wait=False` 로 백그라운드에서, `model2` 는 `wait=True` 로 동기 실행한 뒤,
위 셀 마지막에서 `model1` 의 완료까지 명시적으로 기다립니다. 따라서 이 시점에는 **두 모델 모두 훈련이 끝난 상태**입니다.

- 첫 번째 모델: 보수적 설정(eta=0.5, 50 라운드) — 백그라운드 실행
- 두 번째 모델: 적극적 설정(eta=0.1, 100 라운드) — 동기 실행## ⏳ 두 모델 훈련 대기

`model1` 은 `wait=False` 로 백그라운드에서, `model2` 는 `wait=True` 로 동기 실행한 뒤,
위 셀 마지막에서 `model1` 의 완료까지 명시적으로 기다립니다. 따라서 이 시점에는 **두 모델 모두 훈련이 끝난 상태**입니다.

- 첫 번째 모델: 보수적 설정(eta=0.5, 50 라운드) — 백그라운드 실행
- 두 번째 모델: 적극적 설정(eta=0.1, 100 라운드) — 동기 실행

## 6. MLflow 수동 로깅

현재 실행 중인 MLflow 실험에 훈련된 모델의 정보를 로깅합니다.

In [11]:
def load_model(model_data_s3_uri):
    """S3에서 훈련된 XGBoost 모델을 로드하는 함수"""
    import xgboost as xgb
    import tarfile
    import pickle as pkl

    model_file = "./xgboost-model.tar.gz"
    bucket, key = model_data_s3_uri.replace("s3://", "").split("/", 1)
    boto3.client("s3").download_file(bucket, key, model_file)
    
    with tarfile.open(model_file, "r:gz") as t:
        t.extractall(path=".")
    
    # 모델 로드
    model = xgb.Booster()
    model.load_model("xgboost-model")

    return model

print("✅ 모델 로딩 함수 정의 완료")

✅ 모델 로딩 함수 정의 완료


In [12]:
# MLflow 실행 상태 확인 및 정리
try:
    # 현재 활성화된 실행이 있는지 확인
    active_run = mlflow.active_run()
    if active_run:
        print(f"⚠️ 활성화된 MLflow 실행 발견: {active_run.info.run_id}")
        if active_run.info.run_id != run_id:
            print("   다른 실행이 활성화되어 있습니다. 종료합니다.")
            mlflow.end_run()
        else:
            print("   현재 실행과 동일합니다. 계속 진행합니다.")
    else:
        print("✅ 활성화된 MLflow 실행이 없습니다.")
except Exception as e:
    print(f"MLflow 상태 확인 중 오류: {e}")
    # 안전하게 모든 실행 종료
    try:
        mlflow.end_run()
    except:
        pass

⚠️ 활성화된 MLflow 실행 발견: b2334fdb9a5249f1a6b5f329847c5500
   현재 실행과 동일합니다. 계속 진행합니다.


In [13]:
# [2026-06-20 수정] 하이퍼파라미터는 위에서 정의한 dict 에서 파생해 로깅,
#   model1/model2 동일하던 메트릭 로깅 로직은 log_auc_metrics() 헬퍼로 통합.
# 현재 활성화된 MLflow 실행에 두 모델의 정보 로깅
# (이미 start_run 으로 활성화된 실행이 있으므로 새 run 없이 직접 로깅)

# 1) 하이퍼파라미터 — 위에서 정의한 dict 를 재사용해 model1_*/model2_* prefix 로 기록
print("📝 하이퍼파라미터 로깅...")
mlflow.log_params({f"model1_{k}": v for k, v in model1_hyperparams.items()})
mlflow.log_params({f"model2_{k}": v for k, v in model2_hyperparams.items()})
mlflow.log_params({
    "model1_training_job": xgb1.latest_training_job.name,
    "model2_training_job": xgb2.latest_training_job.name,
})

# 2) 태그 설정
mlflow.set_tags({
    'mlflow.user': user_profile_name,
    'mlflow.source.type': 'TRAINING_JOB',
    'model.algorithm': 'XGBoost',
    'model.framework': 'Built-in',
    'training.approach': 'hyperparameter_comparison'
})

# 3) 메트릭 로깅 — 두 모델 모두 동일 로직이라 헬퍼로 처리
def log_auc_metrics(estimator, prefix):
    df = estimator.training_job_analytics.dataframe()
    for _, row in df.iterrows():
        if row['metric_name'] in ('train:auc', 'validation:auc'):
            name = row['metric_name'].replace(':', '_')
            mlflow.log_metric(f"{prefix}_{name}", row['value'])

try:
    log_auc_metrics(xgb1, "model1")
    log_auc_metrics(xgb2, "model2")
    print("✅ 메트릭 로깅 완료")
except Exception as e:
    print(f"⚠️ 메트릭 로깅 실패: {e}")

# 4) 모델 아티팩트 로깅
# 로깅이 실패하면 이후 register_model(runs:/.../model1) 이 빈 아티팩트를 등록해
# 3번 노트북 로드에서 실패합니다. 조용히 넘어가지 않고 여기서 중단합니다.
try:
    mlflow.xgboost.log_model(load_model(xgb1.model_data), artifact_path="model1")
    mlflow.xgboost.log_model(load_model(xgb2.model_data), artifact_path="model2")
    print("✅ 모델 아티팩트 로깅 완료")
except Exception as e:
    print(f"❌ 모델 로깅 실패 — 등록을 진행하지 않고 중단합니다: {e}")
    raise

print(f"✅ MLflow 로깅 완료 (Run ID: {run_id})")

📝 하이퍼파라미터 로깅...
✅ 메트릭 로깅 완료
✅ 모델 아티팩트 로깅 완료
✅ MLflow 로깅 완료 (Run ID: b2334fdb9a5249f1a6b5f329847c5500)


## 7. 모델 등록

훈련된 모델들을 MLflow Model Registry에 등록하여 버전 관리를 수행합니다.

In [14]:
# 등록할 모델 이름 설정
registered_model_name = f"bank-marketing-model-{experiment_name.split('-')[-1]}"

print(f"✅ 모델 등록 이름: {registered_model_name}")

✅ 모델 등록 이름: bank-marketing-model-43


In [15]:
# 첫 번째 모델 등록
print("📋 첫 번째 모델을 Model Registry에 등록 중...")

model_uri_1 = f"runs:/{run_id}/model1"  # run_id 사용, model1 경로
registered_model_version_1 = mlflow.register_model(
    model_uri_1, 
    registered_model_name
)

print(f"✅ 첫 번째 모델 등록 완료")
print(f"   - 모델 이름: {registered_model_name}")
print(f"   - 버전: {registered_model_version_1.version}")

📋 첫 번째 모델을 Model Registry에 등록 중...


Registered model 'bank-marketing-model-43' already exists. Creating a new version of this model...
2026/06/20 18:39:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bank-marketing-model-43, version 3


✅ 첫 번째 모델 등록 완료
   - 모델 이름: bank-marketing-model-43
   - 버전: 3


Created version '3' of model 'bank-marketing-model-43'.


In [16]:
# 두 번째 모델 등록
print("📋 두 번째 모델을 Model Registry에 등록 중...")

model_uri_2 = f"runs:/{run_id}/model2"  # run_id 사용, model2 경로
registered_model_version_2 = mlflow.register_model(
    model_uri_2, 
    registered_model_name
)

print(f"✅ 두 번째 모델 등록 완료")
print(f"   - 모델 이름: {registered_model_name}")
print(f"   - 버전: {registered_model_version_2.version}")

📋 두 번째 모델을 Model Registry에 등록 중...


Registered model 'bank-marketing-model-43' already exists. Creating a new version of this model...
2026/06/20 18:39:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bank-marketing-model-43, version 4


✅ 두 번째 모델 등록 완료
   - 모델 이름: bank-marketing-model-43
   - 버전: 4


Created version '4' of model 'bank-marketing-model-43'.


## 8. 훈련 결과 확인

MLflow UI에서 훈련 결과를 확인할 수 있는 링크를 생성합니다.

In [17]:
# 최신 실행 정보 가져오기
latest_runs = mlflow.search_runs(
    experiment_ids=[mlflow.get_experiment_by_name(experiment_name).experiment_id], 
    max_results=2, 
    order_by=["attributes.start_time DESC"]
)

# MLflow UI 링크 생성
presigned_url = sess.sagemaker_client.create_presigned_mlflow_tracking_server_url(
    TrackingServerName=mlflow_name,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
# presigned_url(AuthorizedUrl)에는 ?authToken=... 인증 정보가 들어있으므로 자르면 안 됩니다.
# 인증 토큰을 유지한 채 MLflow UI 해시 라우트(#/experiments/{id})만 뒤에 붙입니다.
mlflow_experiment_link = f"{presigned_url}#/experiments/{experiment_id}"

print("✅ MLflow UI 링크 생성 완료")
print(f"   - 실험 페이지: {mlflow_experiment_link}")

✅ MLflow UI 링크 생성 완료
   - 실험 페이지: https://t-bbrizihhee43.us-west-2.experiments.sagemaker.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjllNGI3NWU4LWQ3MGYtNDdiMi04OTU1LTg4MTJhYTgzYjFmMyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDJNNVFaRXFoaklrY0t5M1pybTk1ZC85d0kzbmlQTURJTHRhb2VGYWlqc1FBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGelZVNHdVU3RuVlRWSVNIUkplRUpoUkRoWldEUkJTV3hPWTFOVGNtaDRZMHRCY1cxcldrTlJRbFJFVDFRNGNHSXlTMFZxWkVWSVVIZEVjbTVsVjBWVlFUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzFPVEF4T0RNM016azFNRFE2YTJWNUx6ZzNabUUxTVdReUxURTRNRGt0TkdVMFl5MWhObVV6TFRRNFpXWTNNelk1WW1NM1lnQzRBUUlCQUhnbFNpMTVHb0ZjRTdwWlFua2xaQVdDZG5iSEwraVQ5Y2dETk5rbG9vWUtTQUZmeXZaWkNmUUtkWVBDNGZiaFdwQTZBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1iQkVNcVBzcGoraEpRR1JCQWdFUWdEc1ZmY1dWU2hhR3hRV1BhN1ViUVhiUzZSOG05bGVldlVlbVFUWVpJM0xrMkdqYUk4STVXd2N6MUdQSHMwclAvL1FWeTZTTy9CazBSeTZQWHdJQUFCQUFCQXIvdmlsNU1ENUtUU2R3RUtXZW0wS3VzQmZJTThGK20

In [18]:
# MLflow UI 자동 열기
print("🌐 MLflow UI를 새 탭에서 열고 있습니다...")
display(Javascript(f'window.open("{mlflow_experiment_link}");'))

print("\n📊 MLflow UI에서 다음을 확인할 수 있습니다:")
print("   - 두 모델의 하이퍼파라미터 비교")
print("   - 훈련/검증 AUC 메트릭 비교")
print("   - 모델 아티팩트 및 메타데이터")
print("   - 훈련 작업 링크")

🌐 MLflow UI를 새 탭에서 열고 있습니다...


<IPython.core.display.Javascript object>


📊 MLflow UI에서 다음을 확인할 수 있습니다:
   - 두 모델의 하이퍼파라미터 비교
   - 훈련/검증 AUC 메트릭 비교
   - 모델 아티팩트 및 메타데이터
   - 훈련 작업 링크


## 9. Script Mode를 사용한 커스텀 훈련

더 세밀한 제어와 자동 로깅을 위해 Script Mode를 사용하여 커스텀 훈련 스크립트로 모델을 훈련합니다.

### Script Mode의 장점
- 커스텀 훈련 로직 구현 가능
- MLflow 자동 로깅 기능 활용
- 분산 훈련 지원
- 더 유연한 하이퍼파라미터 튜닝

> ℹ️ **참고**: 여기서 만드는 Script Mode 모델은 *Script Mode + MLflow autolog 방식을 시연*하기 위한 것입니다.
> 이후 `3-model-evaluation.ipynb` 의 성능 비교는 위에서 만든 Built-in 모델 2개(model1/model2)만 사용하며,
> 이 세 번째 모델은 다운스트림에서 평가/배포하지 않습니다.## 9. Script Mode를 사용한 커스텀 훈련

더 세밀한 제어와 자동 로깅을 위해 Script Mode를 사용하여 커스텀 훈련 스크립트로 모델을 훈련합니다.

### Script Mode의 장점
- 커스텀 훈련 로직 구현 가능
- MLflow 자동 로깅 기능 활용
- 분산 훈련 지원
- 더 유연한 하이퍼파라미터 튜닝

> ℹ️ **참고**: 여기서 만드는 Script Mode 모델은 *Script Mode + MLflow autolog 방식을 시연*하기 위한 것입니다.
> 이후 `3-model-evaluation.ipynb` 의 성능 비교는 위에서 만든 Built-in 모델 2개(model1/model2)만 사용하며,
> 이 세 번째 모델은 다운스트림에서 평가/배포하지 않습니다.

In [19]:
# 훈련 스크립트 디렉토리 생성
!mkdir -p './training'

print("✅ 훈련 스크립트 디렉토리 생성 완료")

✅ 훈련 스크립트 디렉토리 생성 완료


In [20]:
%%writefile './training/requirements.txt'
mlflow==2.13.2
sagemaker-mlflow==0.1.0
sagemaker-studio

Overwriting ./training/requirements.txt


### 커스텀 훈련 스크립트 작성

MLflow 자동 로깅과 SageMaker 환경을 활용하는 훈련 스크립트를 작성합니다.

In [21]:
%%writefile ./training/train.py
# [2026-06-20 수정]
#   - 삼중으로 중복되던 _execute_training 호출을 contextlib.nullcontext 폴백으로 단일화
#   - autolog 와 중복되던 수동 mlflow.log_params(params) 제거 (AUC 메트릭만 로깅)
#   - 미사용 import pandas / sm_training_env / 'as run' 바인딩 제거

import argparse
import contextlib
import json
import logging
import os
import pickle as pkl
import subprocess
import sys

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def install_requirements():
    """필수 패키지 설치"""
    try:
        logger.info("Installing MLflow packages...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mlflow==2.13.2', '--quiet'])
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sagemaker-mlflow==0.1.0', '--quiet'])
        logger.info("MLflow packages installed successfully")
    except Exception as e:
        logger.warning(f"Package installation failed: {e}")
        logger.info("Continuing without MLflow...")

# 패키지 설치
install_requirements()

try:
    from sagemaker_containers import entry_point
    from sagemaker_xgboost_container.data_utils import get_dmatrix
    from sagemaker_xgboost_container import distributed
    from sklearn.metrics import roc_auc_score
    import xgboost as xgb
    import mlflow
    MLFLOW_AVAILABLE = True
except ImportError as e:
    logger.warning(f"Import error: {e}")
    MLFLOW_AVAILABLE = False

from time import gmtime, strftime

def _xgb_train(params, dtrain, dval, evals, num_boost_round, model_dir, is_master, early_stopping_rounds=None):
    """XGBoost 훈련 함수"""
    logger.info(f"Starting XGBoost training with params: {params}")

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        evals=evals,
        num_boost_round=num_boost_round,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False
    )

    # 메트릭 계산
    val_auc = roc_auc_score(dval.get_label(), booster.predict(dval))
    train_auc = roc_auc_score(dtrain.get_label(), booster.predict(dtrain))

    logger.info(f"Training AUC: {train_auc:.4f}")
    logger.info(f"Validation AUC: {val_auc:.4f}")

    # MLflow 로깅 (autolog 가 하이퍼파라미터는 자동 기록하므로 여기선 AUC 메트릭만)
    if MLFLOW_AVAILABLE and os.getenv('MLFLOW_TRACKING_ARN'):
        try:
            mlflow.log_metrics({"validation_auc": val_auc, "train_auc": train_auc})
        except Exception as e:
            logger.warning(f"MLflow logging failed: {e}")

    # SageMaker 메트릭 출력 (#011 = 탭, SageMaker 가 로그에서 정규식으로 수집)
    print(f"[0]#011train-auc:{train_auc}#011validation-auc:{val_auc}")

    # 모델 저장 (master 노드만)
    if is_master:
        model_location = os.path.join(model_dir, 'xgboost-model')
        with open(model_location, 'wb') as f:
            pkl.dump(booster, f)
        logger.info(f"Model saved to: {model_location}")

def parse_args():
    """명령행 인수 파싱"""
    parser = argparse.ArgumentParser()

    # 하이퍼파라미터
    parser.add_argument('--max_depth', type=int, default=3)
    parser.add_argument('--eta', type=float, default=0.1)
    parser.add_argument('--gamma', type=int, default=0)
    parser.add_argument('--min_child_weight', type=float, default=1)
    parser.add_argument('--subsample', type=float, default=1.0)
    parser.add_argument('--colsample_bytree', type=float, default=1.0)
    parser.add_argument('--verbosity', type=int, default=1)
    parser.add_argument('--objective', type=str, default='binary:logistic')
    parser.add_argument('--num_round', type=int, default=100)
    parser.add_argument('--early_stopping_rounds', type=int, default=10)
    parser.add_argument('--tree_method', type=str, default="auto")
    parser.add_argument('--predictor', type=str, default="auto")

    # SageMaker 환경 변수
    parser.add_argument('--output_data_dir', type=str, default=os.environ.get('SM_OUTPUT_DATA_DIR'))
    parser.add_argument('--model_dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    parser.add_argument('--sm_hosts', type=str, default=os.environ.get('SM_HOSTS'))
    parser.add_argument('--sm_current_host', type=str, default=os.environ.get('SM_CURRENT_HOST'))

    return parser.parse_known_args()

def main():
    """메인 함수"""
    try:
        args, _ = parse_args()
        logger.info("Training started")
        logger.info(f"Hyperparameters: eta={args.eta}, max_depth={args.max_depth}, num_round={args.num_round}")

        # 환경 변수
        suffix = strftime('%d-%H-%M-%S', gmtime())
        user_profile_name = os.getenv('USER', 'sagemaker-user')
        experiment_name = os.getenv('MLFLOW_EXPERIMENT_NAME')
        region = os.getenv('REGION')
        mlflow_arn = os.getenv('MLFLOW_TRACKING_ARN')
        use_mlflow = MLFLOW_AVAILABLE and bool(mlflow_arn)

        logger.info(f"Environment - User: {user_profile_name}, Region: {region}")

        # SageMaker 분산 환경 정보
        sm_hosts = json.loads(args.sm_hosts) if args.sm_hosts else ['localhost']
        sm_current_host = args.sm_current_host or 'localhost'

        # 데이터 로드
        logger.info("Loading training data...")
        dtrain = get_dmatrix(args.train, 'CSV')
        dval = get_dmatrix(args.validation, 'CSV')
        watchlist = [(dtrain, 'train'), (dval, 'validation')] if dval is not None else [(dtrain, 'train')]
        logger.info(f"Data loaded - Train: {dtrain.num_row()} rows, Validation: {dval.num_row()} rows")

        # MLflow 설정 (사용 가능한 경우)
        if use_mlflow:
            try:
                mlflow.set_tracking_uri(mlflow_arn)
                mlflow.set_experiment(experiment_name or f"script-training-{suffix}")
                mlflow.xgboost.autolog(log_model_signatures=False, log_datasets=False)
                logger.info("MLflow configured successfully")
            except Exception as e:
                logger.warning(f"MLflow setup failed: {e}")
                use_mlflow = False

        # 훈련 하이퍼파라미터
        train_hp = {
            'max_depth': args.max_depth,
            'eta': args.eta,
            'gamma': args.gamma,
            'min_child_weight': args.min_child_weight,
            'subsample': args.subsample,
            'colsample_bytree': args.colsample_bytree,
            'verbosity': 0,  # 로그 출력 최소화
            'objective': args.objective,
            'eval_metric': 'auc',
            'tree_method': args.tree_method,
            'predictor': args.predictor,
        }
        xgb_train_args = {
            'params': train_hp,
            'dtrain': dtrain,
            'dval': dval,
            'evals': watchlist,
            'num_boost_round': args.num_round,
            'model_dir': args.model_dir,
            'early_stopping_rounds': args.early_stopping_rounds,
        }

        # MLflow run 안에서 훈련 (불가하면 run 없이).
        # run 시작 실패 시 nullcontext 로 폴백해 _execute_training 호출을 한 곳으로 통일.
        run_cm = contextlib.nullcontext()
        if use_mlflow:
            try:
                run_cm = mlflow.start_run(
                    run_name=f"script-mode-training-{suffix}",
                    description="SageMaker Script Mode XGBoost Training",
                )
            except Exception as e:
                logger.warning(f"MLflow run start failed: {e}")

        with run_cm:
            if use_mlflow and mlflow.active_run() is not None:
                mlflow.set_tags({
                    'mlflow.user': user_profile_name,
                    'mlflow.source.type': 'TRAINING_JOB',
                    'model.framework': 'Script-Mode',
                    'training.mode': 'script',
                })
            _execute_training(sm_hosts, sm_current_host, dtrain, xgb_train_args)

        logger.info("Training completed successfully")

    except Exception as e:
        logger.error(f"Training failed: {str(e)}")
        raise e

def _execute_training(sm_hosts, sm_current_host, dtrain, xgb_train_args):
    """훈련 실행 함수 (단일 노드 / 분산 분기)"""
    if len(sm_hosts) > 1:
        logger.info(f"Distributed training with {len(sm_hosts)} hosts")
        entry_point._wait_hostname_resolution()
        distributed.rabit_run(
            exec_fun=_xgb_train,
            args=xgb_train_args,
            include_in_training=(dtrain is not None),
            hosts=sm_hosts,
            current_host=sm_current_host,
            update_rabit_args=True
        )
    else:
        logger.info("Single node training")
        if not dtrain:
            raise ValueError("No training data available")
        xgb_train_args['is_master'] = True
        _xgb_train(**xgb_train_args)

def model_fn(model_dir):
    """모델 로드 함수 (추론용)"""
    model_path = os.path.join(model_dir, 'xgboost-model')
    with open(model_path, 'rb') as f:
        return pkl.load(f)

if __name__ == '__main__':
    main()


Overwriting ./training/train.py


### 하이퍼파라미터 및 환경 변수 설정

Script Mode 훈련을 위한 하이퍼파라미터와 환경 변수를 설정합니다.

In [22]:
# Script Mode용 하이퍼파라미터 설정
script_hyperparams = {
    'num_round': 75,
    'max_depth': 4,
    'eta': 0.2,
    'gamma': 1,
    'objective': 'binary:logistic',
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'min_child_weight': 2,
    'early_stopping_rounds': 10,
    'verbosity': 1
}

# 환경 변수 설정 (Project 클래스 활용)
env_variables = {
    'MLFLOW_TRACKING_ARN': arn,  # Project에서 가져온 MLflow ARN
    'MLFLOW_EXPERIMENT_NAME': experiment_name,
    'USER': user_profile_name,
    'REGION': region,
}

print("✅ Script Mode 하이퍼파라미터 설정 완료")
print(f"   - 학습률(eta): {script_hyperparams['eta']}")
print(f"   - 최대 깊이: {script_hyperparams['max_depth']}")
print(f"   - 라운드 수: {script_hyperparams['num_round']}")
print(f"   - MLflow ARN: {env_variables['MLFLOW_TRACKING_ARN']}")

✅ Script Mode 하이퍼파라미터 설정 완료
   - 학습률(eta): 0.2
   - 최대 깊이: 4
   - 라운드 수: 75
   - MLflow ARN: arn:aws:sagemaker:us-west-2:975050309668:mlflow-tracking-server/tracking-server-4gwjxwgbrut11s-6rbrrb5bcww7ww-dev


### Script Mode Estimator 생성

XGBoost Script Mode Estimator를 생성하고 훈련을 실행합니다.

In [23]:
# Script Mode Estimator 생성 (Unified Studio 환경 최적화)
xgb_script_mode = XGBoost(
    entry_point='train.py',  # 새로운 스크립트 사용
    source_dir='./training',
    framework_version="1.7-1",  
    hyperparameters=script_hyperparams,
    role=role,  # Project에서 가져온 역할
    instance_count=1, 
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/script-output/',
    code_location=f's3://{bucket}/{prefix}/code/',  # 코드 업로드 경로 명시
    base_job_name="unified-studio-xgb-training",
    environment=env_variables
)

print("✅ Unified Studio Script Mode Estimator 생성 완료")
print(f"   - 프레임워크 버전: 1.7-1")
print(f"   - 인스턴스 타입: ml.m5.large")
print(f"   - 출력 경로: s3://{bucket}/{prefix}/script-output/")
print(f"   - 코드 업로드 경로: s3://{bucket}/{prefix}/code/")
print(f"   - Unified Studio 최적화 스크립트 사용")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
✅ Unified Studio Script Mode Estimator 생성 완료
   - 프레임워크 버전: 1.7-1
   - 인스턴스 타입: ml.m5.large
   - 출력 경로: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/script-output/
   - 코드 업로드 경로: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/code/
   - Unified Studio 최적화 스크립트 사용


### Script Mode 훈련 실행

커스텀 스크립트를 사용하여 모델을 훈련합니다.

In [24]:
# Script Mode 훈련 실행
print("🚀 Script Mode 훈련 시작...")
print("   - 커스텀 훈련 스크립트 사용")
print("   - MLflow 자동 로깅 활성화")
print("   - 실시간 메트릭 추적")

# 로그 활성화로 디버깅
xgb_script_mode.fit(training_inputs, wait=True, logs=True)

print(f"✅ Script Mode 훈련 완료!")
print(f"   - 훈련 작업: {xgb_script_mode.latest_training_job.name}")
print(f"   - 모델 S3 URI: {xgb_script_mode.model_data}")

🚀 Script Mode 훈련 시작...
   - 커스텀 훈련 스크립트 사용
   - MLflow 자동 로깅 활성화
   - 실시간 메트릭 추적
2026-06-20 18:39:10 Starting - Starting the training job...
2026-06-20 18:39:25 Starting - Preparing the instances for training...
2026-06-20 18:39:48 Downloading - Downloading input data...
2026-06-20 18:40:38 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-20 18:41:50.803 ip-10-38-210-245.us-west-2.compute.internal:8 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-06-20 18:41:50.879 ip-10-38-210-245.us-west-2.compute.internal:8 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-06-20:18:41:51:INFO] Imported framework sagemaker

### Script Mode 결과 확인

Script Mode로 훈련된 모델의 결과를 MLflow UI에서 확인합니다.

In [25]:
# Script Mode 훈련 결과 확인
try:
    # 최신 실행 정보 가져오기
    latest_runs = mlflow.search_runs(
        experiment_ids=[mlflow.get_experiment_by_name(experiment_name).experiment_id], 
        max_results=1, 
        order_by=["attributes.start_time DESC"]
    )
    
    if not latest_runs.empty:
        latest_run_id = latest_runs['run_id'][0]
        
        # MLflow UI 링크 생성
        presigned_url = sess.sagemaker_client.create_presigned_mlflow_tracking_server_url(
            TrackingServerName=mlflow_name,
            ExpiresInSeconds=60,
            SessionExpirationDurationInSeconds=1800
        )['AuthorizedUrl']
        
        experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
        # authToken 유지 (자르면 'Missing Tracking Server ARN' 발생). 해시 라우트만 뒤에 추가
        script_run_link = f"{presigned_url}#/experiments/{experiment_id}/runs/{latest_run_id}"
        
        print("✅ Script Mode 훈련 결과 확인")
        print(f"   - 최신 실행 ID: {latest_run_id}")
        print(f"   - MLflow UI: {script_run_link}")
        
        # MLflow UI 자동 열기
        print("\n🌐 MLflow UI를 새 탭에서 열고 있습니다...")
        display(Javascript(f'window.open("{script_run_link}");'))
        
    else:
        print("⚠️ MLflow 실행 정보를 찾을 수 없습니다.")
        
except Exception as e:
    print(f"⚠️ MLflow 결과 확인 중 오류: {e}")

print("\n📊 Script Mode에서 확인 가능한 정보:")
print("   - 자동 로깅된 하이퍼파라미터")
print("   - 훈련/검증 AUC 메트릭")
print("   - 모델 아티팩트 (자동 저장)")
print("   - 훈련 과정 로그")
print("   - SageMaker 훈련 작업 링크")

✅ Script Mode 훈련 결과 확인
   - 최신 실행 ID: dcab0d75ef394d3ea6c871763c3f7921
   - MLflow UI: https://t-bbrizihhee43.us-west-2.experiments.sagemaker.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6ImMzNTExMjZkLTVjMzItNDdkYS1hNjAzLWIyOTg5MDZiZjEwNiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGxOTGw2bFBHUmw3R0JlMzdIVFl4Wi9Ed3JENWk5TzUxbkZIdUo1TWRsd1lBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGcWJtUm9Za052V1U0cldISlFUbEJUUzJWWlNXVm1PWGNyZEhFNVZqRlBOa28yYldaUlpGQndkekJTV1dWVU1IVXdhVkkxYUdSeGQzSnRhRVYxT1dkNVVUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzFPVEF4T0RNM016azFNRFE2YTJWNUx6ZzNabUUxTVdReUxURTRNRGt0TkdVMFl5MWhObVV6TFRRNFpXWTNNelk1WW1NM1lnQzRBUUlCQUhnbFNpMTVHb0ZjRTdwWlFua2xaQVdDZG5iSEwraVQ5Y2dETk5rbG9vWUtTQUd2NytyTzZtUmI3VCtGSHFvY2NKVzFBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1Wc1AxTllKSStFL3k1SjFtQWdFUWdEc056M29wcEN5TUdBK3lMNkZqbE1zYjI3bUlhcWNDUzZCcUdlT0FoZVl0Q0RFc0ZDL09oblJDYnFOVnMzTWNENWdzTGtiRFZBSEhMcVdIel

<IPython.core.display.Javascript object>


📊 Script Mode에서 확인 가능한 정보:
   - 자동 로깅된 하이퍼파라미터
   - 훈련/검증 AUC 메트릭
   - 모델 아티팩트 (자동 저장)
   - 훈련 과정 로그
   - SageMaker 훈련 작업 링크


## ✅ 모델 훈련 완료

SageMaker Training Job을 사용한 XGBoost 모델 훈련이 성공적으로 완료되었습니다!

### 완료된 작업
- ✅ 두 가지 하이퍼파라미터 설정으로 XGBoost 모델 훈련 (Built-in)
- ✅ MLflow를 통한 실험 추적 및 메트릭 로깅
- ✅ 훈련된 모델을 Model Registry에 등록
- ✅ Script Mode 커스텀 훈련 시연 (autolog)
- ✅ 모델 성능 비교를 위한 MLflow UI 링크 생성

### 훈련된 모델
1. **보수적 모델**: 빠른 학습률(0.5), 적은 라운드(50)
2. **적극적 모델**: 느린 학습률(0.1), 많은 라운드(100)

### 다음 단계
이제 `3-model-evaluation.ipynb` 노트북으로 진행하여:
- 훈련된 두 모델(model1/model2)의 성능 평가 및 비교
- 최적 모델 선택

이어서 `4-test-and-deploy.ipynb` 에서:
- SageMaker 엔드포인트로 모델 배포
- 실시간 추론 테스트## ✅ 모델 훈련 완료

SageMaker Training Job을 사용한 XGBoost 모델 훈련이 성공적으로 완료되었습니다!

### 완료된 작업
- ✅ 두 가지 하이퍼파라미터 설정으로 XGBoost 모델 훈련 (Built-in)
- ✅ MLflow를 통한 실험 추적 및 메트릭 로깅
- ✅ 훈련된 모델을 Model Registry에 등록
- ✅ Script Mode 커스텀 훈련 시연 (autolog)
- ✅ 모델 성능 비교를 위한 MLflow UI 링크 생성

### 훈련된 모델
1. **보수적 모델**: 빠른 학습률(0.5), 적은 라운드(50)
2. **적극적 모델**: 느린 학습률(0.1), 많은 라운드(100)

### 다음 단계
이제 `3-model-evaluation.ipynb` 노트북으로 진행하여:
- 훈련된 두 모델(model1/model2)의 성능 평가 및 비교
- 최적 모델 선택

이어서 `4-test-and-deploy.ipynb` 에서:
- SageMaker 엔드포인트로 모델 배포
- 실시간 추론 테스트

In [26]:
# 훈련 완료 요약
print("🎉 모델 훈련 완료!")
print("=" * 50)

print(f"📊 첫 번째 모델 (보수적 하이퍼파라미터):")
print(f"   - 훈련 작업명: {xgb1.latest_training_job.name}")
print(f"   - 모델 S3 URI: {xgb1.model_data}")
print(f"   - 하이퍼파라미터: {xgb1.hyperparameters()}")

print(f"\n📊 두 번째 모델 (적극적 하이퍼파라미터):")
print(f"   - 훈련 작업명: {xgb2.latest_training_job.name}")
print(f"   - 모델 S3 URI: {xgb2.model_data}")
print(f"   - 하이퍼파라미터: {xgb2.hyperparameters()}")

print(f"\n📜 Script Mode 모델:")
print(f"   - 훈련 작업명: {xgb_script_mode.latest_training_job.name}")
print(f"   - 모델 S3 URI: {xgb_script_mode.model_data}")
print(f"   - MLflow 실행 ID: {run_id}")

print("\n✅ 모든 모델이 성공적으로 훈련되었습니다!")
print("   다음 단계: 3-model-evaluation.ipynb에서 성능 평가")

🎉 모델 훈련 완료!
📊 첫 번째 모델 (보수적 하이퍼파라미터):
   - 훈련 작업명: conservative-xgb-training-2026-06-20-18-35-27-680
   - 모델 S3 URI: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/output/conservative-xgb-training-2026-06-20-18-35-27-680/output/model.tar.gz
   - 하이퍼파라미터: {'max_depth': 3, 'eta': 0.5, 'gamma': 4, 'eval_metric': 'auc', 'min_child_weight': 6, 'subsample': 0.8, 'verbosity': 0, 'objective': 'binary:logistic', 'num_round': 50}

📊 두 번째 모델 (적극적 하이퍼파라미터):
   - 훈련 작업명: aggressive-xgb-training-2026-06-20-18-35-29-287
   - 모델 S3 URI: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/output/aggressive-xgb-training-2026-06-20-18-35-29-287/output/model.tar.gz
   - 하이퍼파라미터: {'max_depth': 3, 'eta': 0.1, 'gamma': 2, 'eval_metric': 'auc', 'min_child_weight': 3, 'subsample': 0.4, 'verbosity': 0, 'objective': 'binary:logistic', 'num_round': 100}

📜 Script Mode 모델:
   - 훈련 작업명: unified-studio-xgb-training-2026-06-20-18-39

In [27]:
# [2026-06-20 수정] 중복되던 두 개의 %store 셀을 이 셀 하나로 통합 (버전 번호 재계산 중복 제거).
#   다운스트림에서 참조되지 않던 run_name_1/run_name_2 (및 생성 셀)도 함께 제거.
# 다음 노트북(3-model-evaluation, 4-test-and-deploy)에서 사용할 변수 일괄 저장
# ModelVersion 객체 대신 버전 번호(문자열)만 저장 → pickle 취약성 회피
registered_model_version_1_num = registered_model_version_1.version
registered_model_version_2_num = registered_model_version_2.version
model1_uri = xgb1.model_data
model2_uri = xgb2.model_data

%store run_id
%store registered_model_name
%store registered_model_version_1_num
%store registered_model_version_2_num
%store model1_uri
%store model2_uri

print("✅ 변수 저장 완료")
print(f"   - MLflow 실행 ID: {run_id}")
print(f"   - 등록된 모델 이름: {registered_model_name}")
print(f"   - 모델 버전: {registered_model_version_1_num}, {registered_model_version_2_num}")
print(f"   - 모델 S3 URI (model1): {model1_uri}")
print(f"   - 모델 S3 URI (model2): {model2_uri}")

Stored 'run_id' (str)
Stored 'registered_model_name' (str)
Stored 'registered_model_version_1_num' (str)
Stored 'registered_model_version_2_num' (str)
Stored 'model1_uri' (str)
Stored 'model2_uri' (str)
✅ 변수 저장 완료
   - MLflow 실행 ID: b2334fdb9a5249f1a6b5f329847c5500
   - 등록된 모델 이름: bank-marketing-model-43
   - 모델 버전: 3, 4
   - 모델 S3 URI (model1): s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/output/conservative-xgb-training-2026-06-20-18-35-27-680/output/model.tar.gz
   - 모델 S3 URI (model2): s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/output/aggressive-xgb-training-2026-06-20-18-35-29-287/output/model.tar.gz
